# Answer Extraction

In [1]:
import json
from pathlib import Path
from evaluator import AnswerExtractor

extractor = AnswerExtractor()

results_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/results")
answers_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/answers")

fail_count = 0
tot = 0

for input_path in results_base.glob("*/*/*.json"):
    dataset = input_path.parts[-3]
    model = input_path.parts[-2]
    filename = input_path.name

    output_path = answers_base / dataset / model / filename

    if output_path.exists():
        continue

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with input_path.open() as f:
        results = json.load(f)

    runs = []
    for run_results in results:

        items = []
        for item in run_results:
            tot = tot+1
            output = item.get("output", {})
            output_text = output.get("output_text") or output.get("text", "")
            finish_reason = output.get("finish_reason")
    
            if dataset == 'gpqa':
                choices = ['A', 'B', 'C', 'D']
            else:
                choices = item.get("options") or item.get("choices")
    
            golden = item.get("answer")
            question = item.get("question") or item.get("problem")
            subject = item.get("subject") or item.get("category")
            logprobs = output.get("logprobs")

            extracted = extractor.extract(
                output_text=output_text,
                choices=choices,
                dataset=dataset,
                finish_reason=finish_reason,
            )

            answer_logprob, cleaned_logprobs = extractor.get_logprobs(extracted, logprobs)

            if extracted.startswith('N.D.'):
                fail_count = fail_count + 1
                # print(f"## OUTPUT ## :{output_text[len(output_text) // 2 :]}\n\n\n")

            items.append({
                "subject": subject,
                "question": question,
                "choices": choices,
                "golden_label": golden,
                "finish_reason": finish_reason,
                'logprobs': cleaned_logprobs,
                "extracted_answer": extracted,
                "extracted_logprob": answer_logprob,
            })

        runs.append(items)

    output_path.write_text(json.dumps(runs, indent=2, ensure_ascii=False) + "\n")

print(f"tot: {tot} fails: {fail_count} ratio: {fail_count/tot}")

[nltk_data] Downloading package stopwords to /home/dario/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Couldn't extract the logprob for answer M and logprobs: [['To', -0.08080260455608368], [' find', -0.4891432821750641], [' the', -6.270212179515511e-05], [' greatest', -0.6146400570869446], [' negative', -3.2066785934148356e-05], [' number', -0.0009076051646843553], [' in', -0.0006725909770466387], [' set', -1.2235145568847656], [' B', -0.000592890428379178], [',', -0.07080010324716568], [' we', -0.07643553614616394], [' need', -1.4195722341537476], [' to', -0.0001530530134914443], [' find', -0.9895312786102295], [' all', -0.7131063938140869], [' possible', -0.8867744207382202], [' values', -1.7626341581344604], [' of', -0.011414814740419388], [' m', -0.2797245979309082], [' and', -1.7064071893692017], [' n', -1.2993727978027891e-05], [' in', -2.5562374591827393], [' set', -0.06449733674526215], [' A', -8.22540732769994e-06], ['.', -3.3610472679138184], [' \n\n', -1.0813965797424316], ['Set', -2.3892717361450195], [' A', -0.0004267973708920181], [' is', -0.30737748742103577], [' defined

Exception: 